# 🧱 SmartLease Edge: YOLOv8n-Seg Multi-Defect Segmentation & Metric Engine
### **iQOO Hackathon 2026 — Chennai City Battle**
**"Cracks become square feet, not opinions."**

---
### 🎯 Project Overview
This notebook trains a lightweight **YOLOv8n-Seg (Instance Segmentation)** model on your newly balanced **2,200+ wall inspection images** across 4 unified defect categories:
1. **`0: crack`** (fine & structural wall cracks — 2,861 instances, ~34%)
2. **`1: peeling`** (paint peeling & surface scaling — 1,678 instances, ~20%)
3. **`2: spalling`** (concrete holes, spall, exposed structural distress — 2,701 instances, ~32%)
4. **`3: stain_mould`** (dampness, moisture induced blackening, efflorescence — 1,150 instances, ~14%)

**Total balanced instances**: **8,390 defect annotations** with a healthy ~1.5:1 to 2.5:1 class ratio (eliminating the old 100:1 crack imbalance!).

### 📐 On-Device Metric Formula (from Slide):
$$\text{Area}_{\text{sq ft}} = \left(\frac{\text{Mask Pixels} \times Z^2}{f_x \times f_y}\right) \times 10.7639$$

### ⚡ Edge Deployment Target:
- Output: **ONNX** / **TFLite** / **Qualcomm Snapdragon Hexagon NPU (ExecuTorch INT8)**
- Targets: $< 30\text{ ms}$ on Snapdragon 8-series, $\sim 6\text{ MB}$ INT8 binary.

--- 
## ⚙️ 1. Hardware Check & Library Installation

In [ ]:
# Verify GPU allocation (T4 / V100 / A100)
!nvidia-smi

In [ ]:
# Install Ultralytics and dependencies for training and edge export
%pip install -q ultralytics opencv-python matplotlib onnx onnxruntime onnxslim

In [ ]:
import os
import sys
import glob
import math
import shutil
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
import torch
import yaml
import ultralytics
from ultralytics import YOLO

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print(f"Ultralytics version: {ultralytics.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

--- 
## 📁 2. Upload & Ingest Dataset

### Instructions:
1. In the Colab left sidebar, click the **Folder (Files)** icon.
2. Drag and drop your **`dataset.zip`** into `/content/` (or place it in your Google Drive at `My Drive/dataset.zip`).
   - Your zip can either contain folders `1`, `2`, `3` OR the pre-merged dataset.
   - The cell below handles extraction and automatic unification automatically!

In [ ]:
# Optional: Mount Google Drive if dataset.zip is stored there
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except Exception as e:
    print("Drive mount skipped or running outside Colab.")

In [ ]:
# Extract dataset.zip
RAW_DIR = Path("/content/raw_data")
RAW_DIR.mkdir(parents=True, exist_ok=True)

colab_zip = Path("/content/dataset.zip")
drive_zip = Path("/content/drive/MyDrive/dataset.zip")

if colab_zip.exists():
    print(f"Extracting {colab_zip} -> {RAW_DIR}...")
    !unzip -q -o /content/dataset.zip -d /content/raw_data/
elif drive_zip.exists():
    print(f"Extracting {drive_zip} -> {RAW_DIR}...")
    !unzip -q -o /content/drive/MyDrive/dataset.zip -d /content/raw_data/
else:
    print("⚠️ Please upload dataset.zip to /content/ using the Colab file browser!")

print("Extracted contents:", [p.name for p in RAW_DIR.iterdir()])

### 🔄 Automated Unification & Class Remapping Engine
This cell merges datasets `1`, `2`, and `3` into our 4 core inspection classes:
- `0: crack`
- `1: peeling`
- `2: spalling`
- `3: stain_mould`
It organizes images and remapped segmentation labels into standard `train`, `valid`, and `test` splits and writes `smartlease_data.yaml`.

In [ ]:
# Unified class taxonomy mapping
MAPPING_1 = {
    3: 0,   # 'Crack' -> crack
    7: 1,   # 'Paint Peeling' -> peeling
    8: 1,   # 'Scaling' -> peeling
    9: 2,   # 'Spalling' -> spalling
    10: 2,  # 'exposed brickwork' -> spalling
    5: 2,   # 'Incomplete Plaster Work' -> spalling
    2: 3,   # 'Corrosion stain' -> stain_mould
    6: 3,   # 'Moss growth due to damping' -> stain_mould
    11: 3,  # 'pollution -moisture induced blackening' -> stain_mould
    1: 3,   # 'Corrosion' -> stain_mould
}

MAPPING_2 = {
    0: 0,   # 'mildcrack' -> crack
    1: 0,   # 'severecrack' -> crack
    2: 2,   # 'spall' -> spalling
}

MAPPING_3 = {
    0: 1,   # 'Paint peeling' -> peeling
    1: 2,   # 'Spalling' -> spalling
    3: 0,   # 'crack' -> crack
    4: 3,   # 'dampness' -> stain_mould
    5: 3,   # 'efflorescence' -> stain_mould
    2: 3,   # 'corrosion' -> stain_mould
}

UNIFIED_DIR = Path("/content/unified_wall_defects")
for split in ["train", "valid", "test"]:
    (UNIFIED_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (UNIFIED_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

# Check if raw_data already has pre-merged data.yaml or folders 1, 2, 3
datasets_to_process = []
for folder_name, mapping in [("1", MAPPING_1), ("2", MAPPING_2), ("3", MAPPING_3)]:
    candidates = list(RAW_DIR.glob(f"**/{folder_name}"))
    if candidates:
        datasets_to_process.append((folder_name, candidates[0], mapping))

unified_stats = Counter()
total_images = 0

if datasets_to_process:
    print(f"Found {len(datasets_to_process)} raw sub-datasets (1, 2, 3). Running remapping & unification...")
    for d_name, d_path, mapping in datasets_to_process:
        print(f"  Merging dataset {d_name}...")
        for split in ["train", "valid", "test"]:
            img_dir = d_path / split / "images"
            lbl_dir = d_path / split / "labels"
            if not img_dir.exists(): continue
            
            for img_p in list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpeg")):
                lbl_p = lbl_dir / (img_p.stem + ".txt")
                if not lbl_p.exists(): continue
                
                remapped_lines = []
                with open(lbl_p, "r") as f:
                    for line in f:
                        parts = line.strip().split()
                        if parts:
                            old_cid = int(parts[0])
                            if old_cid in mapping:
                                new_cid = mapping[old_cid]
                                remapped_lines.append(f"{new_cid} {' '.join(parts[1:])}\n")
                                unified_stats[new_cid] += 1
                                
                if remapped_lines:
                    out_img_name = f"d{d_name}_{img_p.name}"
                    out_lbl_name = f"d{d_name}_{img_p.stem}.txt"
                    shutil.copy(img_p, UNIFIED_DIR / split / "images" / out_img_name)
                    with open(UNIFIED_DIR / split / "labels" / out_lbl_name, "w") as fo:
                        fo.writelines(remapped_lines)
                    total_images += 1
else:
    print("Using pre-extracted unified dataset...")
    UNIFIED_DIR = RAW_DIR

# Create YAML config
CLASS_NAMES = {
    0: "crack",
    1: "peeling",
    2: "spalling",
    3: "stain_mould"
}

yaml_config = {
    "path": str(UNIFIED_DIR.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": CLASS_NAMES
}

yaml_path = UNIFIED_DIR / "smartlease_data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yaml_config, f, default_flow_style=False)

print("\n" + "="*60)
print("🎉 UNIFIED DATASET READY!")
print(f"YAML Path: {yaml_path}")
print(f"Total Images Processed: {total_images:,}")
print("="*60)

--- 
## 📊 3. Exploratory Data Analysis (EDA) & Balance Verification
Let's visualize the class distribution and inspect sample wall images with their corresponding segmentation polygon overlays.

In [ ]:
# Calculate and plot balanced class distribution
counts_by_class = Counter()
for lbl_file in UNIFIED_DIR.glob("**/labels/*.txt"):
    with open(lbl_file, "r") as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                cid = int(parts[0])
                if cid in CLASS_NAMES:
                    counts_by_class[cid] += 1

c_ids = sorted(CLASS_NAMES.keys())
c_names = [CLASS_NAMES[i] for i in c_ids]
c_vals = [counts_by_class[i] for i in c_ids]
total_inst = sum(c_vals)

print("=== Class Instance Breakdown ===")
for name, val in zip(c_names, c_vals):
    pct = (val / total_inst * 100) if total_inst > 0 else 0
    print(f"• {name:<12}: {val:,} instances ({pct:.1f}%)")
print(f"Total Defect Instances: {total_inst:,}")

# Bar chart visualization
plt.figure(figsize=(9, 4.5))
colors = ['#f59e0b', '#10b981', '#ef4444', '#06b6d4']
bars = plt.bar(c_names, c_vals, color=colors, edgecolor='black', alpha=0.88)
plt.title("SmartLease Edge: Balanced Defect Instance Distribution (4 Classes)", fontsize=13, fontweight='bold')
plt.ylabel("Instance Count", fontsize=11)
for b in bars:
    h = b.get_height()
    plt.text(b.get_x() + b.get_width()/2.0, h + (max(c_vals)*0.02), f"{h:,}", ha='center', va='bottom', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

### 🔍 Overlaying Real Ground-Truth Segmentation Polygons
Let's verify that the polygons correspond to actual wall surface defects.

In [ ]:
def render_gt_segmentation(num_samples=4):
    train_imgs = list((UNIFIED_DIR / "train" / "images").glob("*.*"))
    if not train_imgs: return
    
    random.shuffle(train_imgs)
    
    color_palette = {
        0: (255, 215, 0),    # Yellow for crack
        1: (255, 140, 0),    # Orange for peeling
        2: (255, 0, 0),      # Red for spalling
        3: (0, 255, 255)     # Cyan for stain/mould
    }
    
    shown = 0
    fig, axes = plt.subplots(num_samples, 2, figsize=(12, 4.5 * num_samples))
    if num_samples == 1: axes = np.expand_dims(axes, axis=0)
    
    for img_p in train_imgs:
        lbl_p = UNIFIED_DIR / "train" / "labels" / (img_p.stem + ".txt")
        if not lbl_p.exists() or os.path.getsize(lbl_p) == 0: continue
        
        img = cv2.imread(str(img_p))
        if img is None: continue
        h, w, _ = img.shape
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        overlay = img_rgb.copy()
        
        with open(lbl_p, "r") as f: lines = f.readlines()
        
        labels_in_image = []
        for line in lines:
            p = list(map(float, line.strip().split()))
            if len(p) >= 6:
                cid = int(p[0])
                coords = np.array(p[1:]).reshape(-1, 2)
                coords[:, 0] *= w
                coords[:, 1] *= h
                pts = coords.astype(np.int32)
                
                col = color_palette.get(cid, (255, 255, 255))
                cv2.fillPoly(overlay, [pts], col)
                cv2.polylines(overlay, [pts], True, (255, 255, 255), 2)
                labels_in_image.append(CLASS_NAMES.get(cid, str(cid)))
                
        blended = cv2.addWeighted(overlay, 0.45, img_rgb, 0.55, 0)
        
        axes[shown, 0].imshow(img_rgb)
        axes[shown, 0].set_title(f"Original Wall Image ({w}x{h})", fontsize=10)
        axes[shown, 0].axis('off')
        
        axes[shown, 1].imshow(blended)
        axes[shown, 1].set_title(f"Ground-Truth: {', '.join(set(labels_in_image))}", fontsize=10, fontweight='bold', color='darkred')
        axes[shown, 1].axis('off')
        
        shown += 1
        if shown >= num_samples: break
        
    plt.suptitle("Sample Wall Inspections with Polygon Ground-Truth Masks", fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

render_gt_segmentation(num_samples=3)

--- 
## 🚀 4. YOLOv8n-Seg Model Training

### Training Strategy:
- **Architecture**: `YOLOv8n-seg.pt` (Nano segmentation model — optimal for edge NPU latency $< 30\text{ ms}$).
- **Augmentations**:
  - `copy_paste=0.3`: Synthetically pastes defect polygons onto other wall backgrounds.
  - `mosaic=1.0`: Multi-scale defect learning.
  - `mixup=0.15`: Blends complex textures.
  - `flipud=0.5, fliplr=0.5`: Robust to orientation.
- **Loss Tuning**:
  - `box=7.5`: Accurate localization.
  - `cls=0.8`: Balanced multi-class penalty.

In [ ]:
# Load pretrained YOLOv8n-seg weights
model = YOLO('yolov8n-seg.pt')
print("Pretrained YOLOv8n-seg loaded successfully.")

In [ ]:
EPOCHS = 50
BATCH_SIZE = 16
IMG_SIZE = 640
PROJECT = "smartlease_vision"
NAME = "yolov8n_seg_balanced_4class"

print(f"Starting training on {yaml_path} for {EPOCHS} epochs...")

results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    workers=4,
    project=PROJECT,
    name=NAME,
    exist_ok=True,
    optimizer='AdamW',
    lr0=0.002,
    lrf=0.01,
    weight_decay=0.0005,
    # Augmentations for balanced learning:
    copy_paste=0.3,
    mosaic=1.0,
    mixup=0.15,
    degrees=10.0,
    flipud=0.5,
    fliplr=0.5,
    close_mosaic=10,
    # Loss tuning:
    box=7.5,
    cls=0.8,
    save=True,
    plots=True,
    device=0 if torch.cuda.is_available() else 'cpu'
)

--- 
## 📈 5. Validation & Quantitative Metrics
Let's evaluate on the held-out validation set and plot the precision, recall, and mAP per defect category.

In [ ]:
# Load best checkpoint
best_weights = Path(f"{PROJECT}/{NAME}/weights/best.pt")
if not best_weights.exists():
    candidates = list(Path(PROJECT).glob("**/weights/best.pt"))
    if candidates: best_weights = candidates[0]

print(f"Evaluating best checkpoint: {best_weights}")
eval_model = YOLO(str(best_weights))
val_metrics = eval_model.val(data=str(yaml_path), split='val')

print("\n" + "="*50)
print("🏆 VALIDATION PERFORMANCE SUMMARY (YOLOv8n-Seg)")
print("="*50)
print(f"Overall Mask mAP@50:    {val_metrics.seg.map50:.4f}")
print(f"Overall Mask mAP@50-95: {val_metrics.seg.map:.4f}")
print(f"Overall Box mAP@50:     {val_metrics.box.map50:.4f}")
print(f"Overall Box mAP@50-95:  {val_metrics.box.map:.4f}")
print("="*50)

In [ ]:
# Plot loss and confusion matrix
train_res_dir = best_weights.parent.parent
results_img = train_res_dir / "results.png"
if results_img.exists():
    plt.figure(figsize=(15, 8))
    plt.imshow(cv2.cvtColor(cv2.imread(str(results_img)), cv2.COLOR_BGR2RGB))
    plt.title("Loss Curves and mAP Progression across Epochs", fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.show()

conf_mat = train_res_dir / "confusion_matrix.png"
if conf_mat.exists():
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(cv2.imread(str(conf_mat)), cv2.COLOR_BGR2RGB))
    plt.title("Confusion Matrix (Multi-Class Defect Classification)", fontsize=12, fontweight='bold')
    plt.axis('off')
    plt.show()

--- 
## 🧪 6. Visual Defect Predictions on Test Images

In [ ]:
test_imgs = list((UNIFIED_DIR / "test" / "images").glob("*.*")) or list((UNIFIED_DIR / "valid" / "images").glob("*.*"))
random.shuffle(test_imgs)

preds = eval_model.predict(source=test_imgs[:6], conf=0.30, imgsz=640)

plt.figure(figsize=(15, 10))
for i, p in enumerate(preds):
    plt.subplot(2, 3, i + 1)
    plt.imshow(cv2.cvtColor(p.plot(), cv2.COLOR_BGR2RGB))
    plt.title(f"Test Wall {i+1}: {len(p.boxes)} Defect(s)", fontsize=11, fontweight='bold')
    plt.axis('off')
plt.suptitle("SmartLease Edge: YOLOv8n-Seg Multi-Defect Segmentation Inferences", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

--- 
## 📐 7. SmartLease Physical Metric Engine (Pinhole Model)
### *"Cracks become square feet, not opinions."*

$$\text{Area}_{\text{m}^2} = \frac{\text{Mask Pixels} \times Z^2}{f_x \times f_y}$$
$$\text{Area}_{\text{sq ft}} = \text{Area}_{\text{m}^2} \times 10.7639$$

### Decision Rules:
- Confidence $< 0.30$: Hidden (Noise)
- Confidence $0.30 - 0.50$: Tagged as `REVIEW`
- Confidence $> 0.50$: Confirmed defect (Minor, Moderate, Severe) + Chennai repair cost estimate

In [ ]:
class SmartLeaseMetricEngine:
    def __init__(self, fx=580.0, fy=580.0, base_rate_sqft=350.0, callout_fee=1200.0):
        self.fx = fx
        self.fy = fy
        self.base_rate = base_rate_sqft
        self.callout = callout_fee
        
    def compute_area(self, mask_pixels, z_meters):
        area_m2 = (mask_pixels * (z_meters ** 2)) / (self.fx * self.fy)
        area_sqft = area_m2 * 10.7639
        return area_m2, area_sqft
        
    def severity(self, area_sqft):
        if area_sqft < 0.10: return "MINOR", (0, 255, 0)
        elif area_sqft <= 0.50: return "MODERATE", (0, 165, 255)
        else: return "SEVERE", (0, 0, 255)
        
    def cost_range(self, area_sqft):
        c = self.callout + (area_sqft * self.base_rate)
        return int(math.floor(c * 0.9 / 50.0) * 50), int(math.ceil(c * 1.15 / 50.0) * 50)

    def evaluate(self, mask_bin, conf, z_meters=1.2):
        if conf < 0.30: return {'status': 'HIDDEN', 'conf': conf}
        st = 'REVIEW' if conf < 0.50 else 'CONFIRMED'
        px = int(np.sum(mask_bin > 0))
        m2, sqft = self.compute_area(px, z_meters)
        sev, col = self.severity(sqft)
        low, high = self.cost_range(sqft)
        return {
            'status': st,
            'confidence': float(conf),
            'pixels': px,
            'area_sqft': round(sqft, 2),
            'area_m2': round(m2, 4),
            'severity': sev,
            'cost': f"₹{low:,} - ₹{high:,}"
        }

engine = SmartLeaseMetricEngine()

# Run engine on sample image with ARCore depth Z = 1.25m
sample_p = test_imgs[0]
pred_out = eval_model.predict(source=str(sample_p), conf=0.25, imgsz=640)[0]
img_orig = pred_out.orig_img.copy()
h, w, _ = img_orig.shape
overlay = img_orig.copy()

Z_dist = 1.25  # Simulated distance in meters
cards = []

if pred_out.masks is not None:
    for i, (box, smask) in enumerate(zip(pred_out.boxes, pred_out.masks.data)):
        conf = float(box.conf[0])
        cid = int(box.cls[0])
        cname = CLASS_NAMES[cid]
        
        m_np = cv2.resize(smask.cpu().numpy(), (w, h), interpolation=cv2.INTER_NEAREST)
        bin_m = (m_np > 0.5).astype(np.uint8)
        
        res = engine.evaluate(bin_m, conf, z_meters=Z_dist)
        if res['status'] == 'HIDDEN': continue
        
        res['class'] = cname
        res['idx'] = i + 1
        cards.append(res)
        
        # Draw contours
        cnts, _ = cv2.findContours(bin_m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay, cnts, -1, (0, 140, 255), -1)
        cv2.drawContours(img_orig, cnts, -1, (0, 255, 255), 2)
        
        x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
        tag = f"{cname.upper()} · {res['area_sqft']} sq ft [{res['severity']}]"
        cv2.putText(img_orig, tag, (x1, max(25, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 2)

comp = cv2.addWeighted(overlay, 0.35, img_orig, 0.65, 0)
plt.figure(figsize=(9, 7))
plt.imshow(cv2.cvtColor(comp, cv2.COLOR_BGR2RGB))
plt.title("SmartLease Edge AR Scan (Simulated ARCore Depth Z=1.25m)", fontsize=13, fontweight='bold')
plt.axis('off')
plt.show()

print("\n" + "="*60)
print("📋 SMARTLEASE INSPECTION REPORT")
print("="*60)
for c in cards:
    print(f"• Defect #{c['idx']} [{c['class'].upper()}]:")
    print(f"  Status:         {c['status']} (Confidence: {c['confidence']:.2f})")
    print(f"  Physical Area:  {c['area_sqft']} sq ft ({c['area_m2']} m²)")
    print(f"  Severity:       {c['severity']}")
    print(f"  Est. Repair:    {c['cost']} (Chennai plaster + paint)")
    print("-"*60)

--- 
## 📦 8. On-Device Edge Export & Bundle Download

In [ ]:
# 1. Export to ONNX
print("Exporting fine-tuned model to ONNX...")
onnx_path = eval_model.export(format='onnx', opset=12, dynamic=False, simplify=True)
print(f"ONNX Model: {onnx_path}")

In [ ]:
# 2. Package Deployment Bundle
bundle_dir = Path("/content/smartlease_edge_bundle")
bundle_dir.mkdir(parents=True, exist_ok=True)

if best_weights.exists(): shutil.copy(best_weights, bundle_dir / "best.pt")
if Path(onnx_path).exists(): shutil.copy(onnx_path, bundle_dir / "best.onnx")

for chart_name in ["results.png", "confusion_matrix.png", "F1_curve.png", "PR_curve.png"]:
    src = train_res_dir / chart_name
    if src.exists(): shutil.copy(src, bundle_dir / chart_name)

# Zip bundle
!zip -r /content/smartlease_edge_trained_bundle.zip /content/smartlease_edge_bundle/
print("\n✅ Deployment bundle created: /content/smartlease_edge_trained_bundle.zip")

# Trigger 1-click download
try:
    from google.colab import files
    files.download('/content/smartlease_edge_trained_bundle.zip')
    print("Download initiated in browser.")
except Exception as e:
    print("File available in Colab file tree.")